# scratch.ipynb
R3: KNN, Gaussian Naive Bayes and the SVM decision function implemented in NumPy, checked against scikit-learn. Needs `best_params.json`, written by `comparison.ipynb`. Uses X_test only to compare against scikit-learn, never to score or select.

In [1]:
import json, numpy as np
from scipy.special import logsumexp
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

SEED = 42
d = load_breast_cancer(as_frame=True)
X, y = d.data, 1 - d.target          # 1 = malignant
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

best = json.load(open("best_params.json"))   # tuned in comparison.ipynb (same split, same seed)
base = {"knn": KNeighborsClassifier(), "gnb": GaussianNB(), "svm": SVC(random_state=SEED)}
fit = {}
for n, c in base.items():
    fit[n] = Pipeline([("scale", StandardScaler()), ("clf", c)]).set_params(**best[n]).fit(X_train, y_train)
print({n: best[n] for n in base})

sc = fit["knn"].named_steps["scale"]
Xtr, Xte = sc.transform(X_train), sc.transform(X_test)   # scaled as in the pipeline
ytr = y_train.to_numpy()

{'knn': {'clf__n_neighbors': 21, 'clf__p': 2, 'clf__weights': 'distance'}, 'gnb': {'clf__var_smoothing': 1.318078594935307e-08}, 'svm': {'clf__C': 110.15056790269621, 'clf__gamma': 0.001653693718282443, 'clf__kernel': 'rbf'}}


## KNN, vectorised (no Python loop over training rows)
Distance is Minkowski with the tuned `p`; weights follow scikit-learn (exact matches dominate under distance weighting).

In [2]:
def knn_predict(Xtr, ytr, Xte, k, p, weights):
    D = (np.abs(Xte[:, None, :] - Xtr[None, :, :]) ** p).sum(-1) ** (1 / p)
    idx = np.argpartition(D, k - 1, axis=1)[:, :k]
    dd = np.take_along_axis(D, idx, 1)
    if weights == "uniform":
        w = np.ones_like(dd)
    else:
        z = dd == 0
        w = np.where(z.any(1, keepdims=True), z.astype(float), 1 / np.where(z, 1, dd))
    votes = (w[..., None] * np.eye(2)[ytr[idx]]).sum(1)
    return votes.argmax(1)

knn = fit["knn"].named_steps["clf"]
mine = knn_predict(Xtr, ytr, Xte, knn.n_neighbors, knn.p, knn.weights)
print(knn.n_neighbors, knn.p, knn.weights, "| predictions match on all", len(Xte), "test rows:",
      np.array_equal(mine, knn.predict(Xte)))

21 2 distance | predictions match on all 114 test rows: True


## Gaussian Naive Bayes, in log space, with var_smoothing
scikit-learn adds `var_smoothing * max(variance of each feature over all training rows)` to every class variance.

In [3]:
def gnb_fit(X, y, var_smoothing):
    classes = np.unique(y)
    eps = var_smoothing * X.var(axis=0).max()
    theta = np.array([X[y == c].mean(0) for c in classes])
    var = np.array([X[y == c].var(0) for c in classes]) + eps
    prior = np.array([(y == c).mean() for c in classes])
    return theta, var, prior

def gnb_predict_proba(X, theta, var, prior):
    jll = (np.log(prior)
           - 0.5 * np.log(2 * np.pi * var).sum(1)
           - 0.5 * (((X[:, None, :] - theta) ** 2) / var).sum(2))
    return np.exp(jll - logsumexp(jll, axis=1, keepdims=True))

gnb = fit["gnb"].named_steps["clf"]
theta, var, prior = gnb_fit(Xtr, ytr, gnb.var_smoothing)
print("var_smoothing:", gnb.var_smoothing, "| np.allclose to GaussianNB.predict_proba:",
      np.allclose(gnb_predict_proba(Xte, theta, var, prior), gnb.predict_proba(Xte)))

var_smoothing: 1.318078594935307e-08 | np.allclose to GaussianNB.predict_proba: True


## SVM decision function from support_vectors_, dual_coef_ and intercept_

In [4]:
def svm_decision(X, svm):
    SV = svm.support_vectors_
    if svm.kernel == "linear":
        K = X @ SV.T
    elif svm.kernel == "rbf":
        sq = (X ** 2).sum(1)[:, None] + (SV ** 2).sum(1)[None, :] - 2 * X @ SV.T
        K = np.exp(-svm.gamma * sq)
    else:
        raise ValueError(svm.kernel)
    return K @ svm.dual_coef_.ravel() + svm.intercept_[0]

svm = fit["svm"].named_steps["clf"]
print(svm.kernel, "C =", svm.C, "gamma =", svm.gamma, "| np.allclose to SVC.decision_function:",
      np.allclose(svm_decision(Xte, svm), svm.decision_function(Xte)))

rbf C = 110.15056790269621 gamma = 0.001653693718282443 | np.allclose to SVC.decision_function: True
